In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

## 1. Data exploration


In [ ]:
df = pd.read_csv('data/creditcard.csv')

df.head(10)


In [ ]:
df.info()

In [ ]:
print("Shape:", df.shape)
print("Missing Values:", df.isna().sum().sum())
print("Dupe Values:", df.duplicated().sum())
print("Fraud rate: {:.3%}".format(df["Class"].mean()))
print(df[["Time", "Amount"]].describe())

In [ ]:
df.head(28000)

In [ ]:
plt.figure(figsize =(15,12))
sns.boxplot(x = 'Amount', data = df, color = "blue")
plt.ylabel("All transactions")
plt.title("Amount outliers")
plt.show()

In [ ]:
plt.figure(figsize =(15,12))
sns.boxplot(x = 'Amount', y = 'Class', data = df, orient = "h")
plt.yticks([0,1], ["Legitimate", "Fraud"])

## 2. Data cleaning

In [ ]:
n_dupes = df.duplicated().sum()
n_fraud_dupes = df[df.duplicated()]["Class"].sum()
print("Duplicates:", n_dupes, "of which fraud:", n_fraud_dupes)

df = df.drop_duplicates().reset_index(drop=True)
print("Rows after cleaning:", len(df), "| frauds:", df["Class"].sum())

## 3. Feature engineering


In [ ]:
df["Hour"] = (df['Time'] // 3600) % 24
#amount is skewed so we can use log to fix this(ai reccommendation)
df["Amount_log"] = np.log1p(df["Amount"])
df = df.drop(columns = ['Time', 'Amount'])

#dropping time can create a data leak as we are replacing it with hour 
before = len(df)
df = df.drop_duplicates().reset_index(drop = True)
print("Extra duplicates removed:", before - len(df), "| frauds kept:", df["Class"].sum())

## 4. Data partitioning

In [ ]:
X = df.drop(columns = 'Class')
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify=y, random_state= 42)
#reminders:
#X_train + y_train	Used to teach the model
#X_test + y_test	Used to check the model on unseen data

print("Train:", X_train.shape, "| fraud rate: {:.3%}".format(y_train.mean()))
print("Test: ", X_test.shape, "| fraud rate: {:.3%}".format(y_test.mean()))


overlap = pd.merge(X_train.drop_duplicates(), X_test.drop_duplicates(), how="inner")
print("Rows shared between train and test:", len(overlap))

## 5. Feature scaling

In [ ]:
scale_column = ['Amount_log', 'Hour']
scaler = StandardScaler()

X_train[scale_column] =  scaler.fit_transform(X_train[scale_column])
X_test[scale_column] =  scaler.transform(X_test[scale_column])

X_train[scale_column].describe().loc[["mean", "std"]] #ai recc



Mean = 0; average is now 0 for each column, its now meassured in above or below avg. Positive means above avg and negative means below avg
Std = 1: a typical row sits about 1 unit from the average. A value of +2 is unusually high, and -2 is unusually low.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter = 1000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)

rf = RandomForestClassifier(n_jobs = -1, class_weight = 'balanced', max_depth = 5, random_state = 42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

for name, pred in [('Logistic regression', lr_pred), ('Random forest', rf_pred)]:
    print(f"\n{name}")
    print(classification_report(y_test, pred))